# SVM(Support Vector Machine) — 분류(SVC) · 회귀(SVR)

수업 중 작성한 필기 `SVM_분류실습` 과 개념 메모 `notbook04.py` 의 SVM 부분을 **실행 가능하도록 정리한 노트**입니다.
(교재 `머신러닝` p.57~65 — 주요 분류 모델: SVM)

메모에서 맞는 내용은 살리고, 틀리게 적힌 부분은 교재와 실행 결과로 확인해 고쳤습니다.
원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

> 실습 데이터 `titanic_cleaned.csv` (Day 022 전처리 결과) · `house_price.csv` (보스턴 주택 가격) 는 이 노트북과 **같은 폴더**에 있어야 합니다.
> 결정 트리는 [`decision_tree.ipynb`](./decision_tree.ipynb) 에 따로 정리했습니다.

## 1. SVM 이란? (교재 p.57~58)

**클래스 간의 안전한 거리(마진, Margin)를 최대화하여 모델의 성능을 높이는 분류 알고리즘.**

```
   ●  ●                     ■  ■
 ●   (●) ┊ - - - ┃ - - - ┊ (■)  ■
   ●     ┊       ┃       ┊    ■
         ┊◀ 마진 ▶┃◀ 마진 ▶┊
       서포트 벡터   결정 경계   서포트 벡터
```

| 용어 | 뜻 |
|------|-----|
| **결정 경계** (Decision boundary) | 두 클래스를 나누는 선 (특성이 많으면 면 = 초평면) |
| **서포트 벡터** (Support Vector) | 결정 경계와 **가장 가까운** 위치의 데이터 — 경계를 "받치는" 점 |
| **마진** (Margin) | 결정 경계와 서포트 벡터 사이의 **거리** |

**목표** — 데이터를 두 클래스로 완벽하게 분리하면서도 **마진이 최대**가 되는 결정 경계를 찾는다.

### 왜 마진을 넓혀야 할까?

학습용 데이터만 보면 한쪽 클래스에 **바짝 붙은** 경계로도 완벽하게 나눌 수 있다.
그런데 평가용(처음 보는) 데이터는 학습용과 조금씩 다른 위치에 있어서, 경계에 바짝 붙어 있으면 **조금만 어긋나도 틀린다.**
그래서 양쪽에서 **최대한 멀리 떨어진(안전거리를 둔)** 경계를 고른다.

| 장점 | 단점 |
|------|------|
| **소규모 데이터에 강함** — 수십~수백 개로도 일반화 능력이 좋다 | **연산 복잡도** — 데이터가 커지면 학습 속도가 급격히 느려진다 |
| **고차원 효율성** — 특성 수가 샘플 수보다 많아도 안정적 | |

| 용도 | 모델 | 평가 |
|------|------|------|
| **분류** | `SVC` (Support Vector **Classifier**) | `accuracy_score` |
| **회귀** | `SVR` (Support Vector **Regressor**) | `root_mean_squared_error` |

> SVM 은 **거리**로 경계를 정하기 때문에 특성마다 범위가 다르면 큰 숫자 특성이 거리를 지배한다 → **표준화 필수.** (Day 020 스케일링 가이드 : SVM → `StandardScaler`)

> **📝 고친 부분** (`notbook04.py` 메모)
> - "결정 경계 한쪽에만 붙을 수 있다 → 평가용에서는 다르다 → 여유 있게 떨어질 필요(안전거리)" → **맞다.** 마진을 최대화하는 이유 그대로
> - "그래서 나온 게 (서포트 **팩터**)" → 서포트 **벡터**. 그리고 서포트 벡터는 "안전거리" 자체가 아니라 **경계와 가장 가까운 데이터**이고, 안전거리는 **마진**이다
> - "마진 : 결정 경계 **사이의** 거리" → 결정 경계와 **서포트 벡터 사이의** 거리
> - "SVM : 데이터는 **많을 때** 유리하다" → 반대다. **적은 데이터(수십~수백 개)에 강하고**, 데이터가 많으면 느려진다
> - "분류 : SVC, 회귀 : **SVM**" → 회귀는 **SVR**. SVM 은 둘을 아우르는 알고리즘 이름
> - "표준화 : 선형 회귀, 로지스틱 회귀, SVM 시켜야 한다" → 맞다. 특히 SVM 은 거리 기반이라 결과가 크게 달라진다 (아래 4단원에서 확인)

## 2. SVC 분류 실습 — 타이타닉 생존 여부 예측 (교재 p.59~61)

```python
from sklearn.svm import SVC          # 1. 임포트
                                     # 2. 데이터 전처리 (Day 022 에서 완료 → titanic_cleaned.csv)
                                     # 3. 학습용 / 평가용 데이터 생성
                                     # 4. 표준화
svc = SVC()                          # 5. 모델 생성
svc.fit(X_train_scaled, y_train)     # 6. 모델 학습
pred_test = svc.predict(X_test_scaled)          # 7. 예측
accuracy = accuracy_score(y_test, pred_test)    # 8. 평가
```

### 2-1. 데이터 불러오기 · 분할

In [ ]:
import pandas as pd

# Day 022 에서 전처리를 끝낸 타이타닉 데이터 (노트북과 같은 폴더)
path = './titanic_cleaned.csv'
df = pd.read_csv(path)

print("모양 :", df.shape)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# 특성(X) : 정답 컬럼을 뺀 나머지 / 정답(y) : Survived
X_data = df.drop(columns=['Survived'])
y_data = df['Survived']

# 정답 비율 → 이 비율을 학습용 · 평가용에서도 유지(stratify)
print(f"정답 비율 : \n{y_data.value_counts(normalize=True).round(3)}")

# 학습용 80 : 평가용 20
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=0,      # 섞는 방식 고정 → 결과 재현
    stratify=y_data      # 정답 비율 유지
)

print("-" * 40)
print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)
print("학습용 정답 비율 :", y_train.value_counts(normalize=True).round(3).tolist())

### 2-2. 표준화 — 평가용은 `transform` 만!

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 학습용 : 평균 · 표준편차 계산(fit) + 변환
X_train_scaled = scaler.fit_transform(X_train)

# 평가용 : 학습용의 기준으로 변환만 (fit_transform ✕)
X_test_scaled = scaler.transform(X_test)

> **📝 고친 부분** — 필기에서 가장 중요한 버그
> - `X_test_scaled = scaler.`**`fit_transform`**`(X_test)` → **`transform(X_test)`**
>   평가용에 `fit_transform` 을 쓰면 평가용 **자기 자신의** 평균 · 표준편차로 다시 기준을 잡는다.
>   학습용과 평가용의 "0" 이 서로 다른 위치를 가리키게 되고, 평가용 정보를 미리 본 셈(데이터 누수)이 된다.
> - 주석 "#학습용 데이터를 기준으로 평균**화** 표준편차" → 평균**과** 표준편차

### 2-3. SVC 모델 학습 · 예측 · 평가

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 모델 생성 (학습 전)
svc = SVC()

# 모델 학습 : 마진이 최대인 결정 경계 찾기
svc.fit(X_train_scaled, y_train)
print("학습 완료!")

# 서포트 벡터 : 경계를 받치는 데이터가 몇 개인가
print("서포트 벡터 수 (클래스 0, 1) :", svc.n_support_, "→ 총", len(svc.support_), "개")

In [ ]:
# 평가용 데이터로 예측
pred_test = svc.predict(X_test_scaled)

# 정확도 = 맞힌 개수 / 평가용 데이터 개수
accuracy_svc = accuracy_score(y_test, pred_test)
print(f"평가용 {len(y_test)}명 중 {(pred_test == y_test).sum()}명 맞힘 → 정확도 {accuracy_svc:.4f}")

# 학습용 정확도와 비교 → 차이가 작으면 과대적합이 심하지 않다
print(f"학습용 정확도 : {accuracy_score(y_train, svc.predict(X_train_scaled)):.4f}")

> **결과 해석** — 정확도 **0.8588 (85.9%)**, 학습용 86.8% 와 차이가 작다.
> 필기의 **0.8626** 은 평가용을 `fit_transform` 한 버그 때문에 나온 값이다. 점수는 조금 높게 나왔지만 **올바르지 않은 방법**이라 0.8588 이 진짜 성능이다.
> Day 022 의 로지스틱 회귀(0.8588)와 같은 수준이다.

> **📝 고친 부분**
> - 쓰지 않는 `from sklearn.linear_model import LinearRegression` 과 `''' … '''` 로 막아 둔 선형 회귀 학습 코드 제거 — 생존 여부(0/1)는 **분류** 문제라 선형 회귀가 아니다
> - `print(X_train.index)` 에 "#학습용 데이터 **개수** 확인", `print(y_train.index)` 에 "#**평가용** 데이터 개수 확인" → 둘 다 **학습용**이고, 출력한 건 개수가 아니라 **행 번호**다. 개수는 `.shape` / `len()`
> - 주석 "random_**sate**, **strafity**" → `random_state`, `stratify`. `stratify` 는 무작위성 통제가 아니라 **정답 비율 유지**
> - `path = r"C:\Users\Jinho\work\titanic_cleaned.csv"` → `./titanic_cleaned.csv`, `warnings.filterwarnings('ignore')` 제거

## 3. SVR 회귀 실습 — 보스턴 주택 가격 예측 (교재 p.62~65) ✚ 필기에 없어 추가

사용법은 SVC 와 같고, 모델과 평가 지표만 다르다.

```python
from sklearn.svm import SVR
svr = SVR()
svr.fit(X_train_scaled, y_train)
pred_test = svr.predict(X_test_scaled)

from sklearn.metrics import root_mean_squared_error as rmse
rmse(y_test, pred_test)
```

> SVR 은 "마진 **안쪽**에 최대한 많은 데이터가 들어오는 띠(튜브)" 를 찾는다. 띠 안의 작은 오차는 봐주고, 띠 밖으로 벗어난 데이터만 손실로 친다.

In [ ]:
# 보스턴 주택 가격 (특성 13개, 506개) — Day 021 과 같은 데이터, 같은 분할
df_house = pd.read_csv('./house_price.csv')

X_data = df_house.drop(columns=['medv'])
y_data = df_house['medv']

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.25, random_state=0
)

# 표준화 : 학습용 기준
scaler_h = StandardScaler()
X_train_scaled = scaler_h.fit_transform(X_train)
X_test_scaled = scaler_h.transform(X_test)

print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error as rmse

# 모델 생성 · 학습 · 예측
svr = SVR()
svr.fit(X_train_scaled, y_train)
pred_test = svr.predict(X_test_scaled)

print(f"SVR RMSE (기본값) : {rmse(y_test, pred_test):.2f}")

In [ ]:
# 하이퍼파라미터 C : 띠 밖으로 벗어난 오차에 주는 벌점의 크기 (기본값 1.0)
#   작을수록 → 오차를 너그럽게 봐준다(단순한 모델) / 클수록 → 오차를 엄격하게 줄인다
for C in [1, 10, 100]:
    model = SVR(C=C).fit(X_train_scaled, y_train)
    print(f"C = {C:>3} → RMSE {rmse(y_test, model.predict(X_test_scaled)):.2f}")

print("-" * 40)
print("참고 : Day 021 LinearRegression RMSE = 5.46")

> **결과 해석**
> - 기본값 SVR 은 RMSE **6.34** 로 선형 회귀(5.46)보다 나빴다.
> - `C` 를 키우니 **10 → 5.01, 100 → 4.19** 로 선형 회귀보다 좋아졌다. SVM 계열은 **하이퍼파라미터에 따라 성능 차이가 크다.**
> - 다만 `C` 를 평가용 점수를 보고 고르면 평가용에 맞춰진 셈이 된다. 제대로 하려면 교차 검증으로 고른다.

## 4. 표준화를 안 하면? — SVM 은 거리 기반

필기 메모 "표준화 : 선형 회귀, 로지스틱 회귀, SVM 시켜야 한다" 를 직접 확인해 본다.

In [ ]:
# 타이타닉 : 표준화 없이 SVC
X_t = df.drop(columns=['Survived'])
y_t = df['Survived']
Xa, Xb, ya, yb = train_test_split(X_t, y_t, test_size=0.2, random_state=0, stratify=y_t)

acc_raw = accuracy_score(yb, SVC().fit(Xa, ya).predict(Xb))
print(f"SVC  표준화 X : {acc_raw:.4f}   /   표준화 O : {accuracy_svc:.4f}")

# 보스턴 : 표준화 없이 SVR
rmse_raw = rmse(y_test, SVR().fit(X_train, y_train).predict(X_test))
print(f"SVR  표준화 X : RMSE {rmse_raw:.2f}   /   표준화 O : RMSE {rmse(y_test, pred_test):.2f}")

> - SVC 는 표준화를 빼자 정확도가 **85.9% → 67.6%** 로 떨어졌다. "전부 사망" 으로 찍는 기준선(62.2%)과 거의 차이가 없다.
> - SVR 도 RMSE 가 **6.34 → 8.65** 로 나빠졌다.
> - `Fare`(최대 66), `tax`(최대 711) 처럼 숫자가 큰 특성이 거리를 독차지하기 때문이다. Day 021 의 선형 회귀는 표준화 여부와 상관없이 결과가 같았던 것과 대조적이다.

## 5. 오늘 정리 (SVM)

| | SVC (분류) | SVR (회귀) |
|---|---|---|
| 임포트 | `from sklearn.svm import SVC` | `from sklearn.svm import SVR` |
| 생성 · 학습 | `SVC().fit(X_train_scaled, y_train)` | `SVR().fit(X_train_scaled, y_train)` |
| 평가 | `accuracy_score` → **0.8588** | `root_mean_squared_error` → **6.34** (C=100 이면 4.19) |

- SVM = 클래스 사이 **마진(안전거리)을 최대화**하는 결정 경계 찾기. 경계에 가장 가까운 점이 **서포트 벡터**.
- **적은 데이터 · 많은 특성**에 강하고, 데이터가 많으면 느리다.
- 거리 기반이라 **표준화 필수** — 안 하면 정확도 85.9% → 67.6%.
- 표준화는 학습용 `fit_transform`, 평가용 **`transform`**.